# 스택(Stack)과 재귀(Recursion) 정리 노트

- **오늘 목표**
  - 스택의 핵심 동작(Push/Pop/Top)을 말로 설명하고, 파이썬으로 직접 구현할 수 있다.
  - 스택이 **괄호 검사**, **함수 호출(시스템 스택)**, **재귀**와 어떻게 연결되는지 이해한다.
  - 피보나치 예제로 **재귀의 중복 호출 문제**를 보고, Memoization/DP로 개선할 수 있다.

- **키워드**: LIFO, push/pop, top, overflow/underflow, stack frame, recursion, memoization, DP


## 오늘의 한 줄

스택은 **“맨 위에 올려둔 걸 제일 먼저 꺼내는 상자”**예요.  
접시를 쌓아두면 *맨 위 접시*부터 꺼내게 되죠. 그 느낌이 그대로 스택입니다.

- **push**: 위에 올리기  
- **pop**: 위에서 빼기  
- **top(또는 peek)**: 위에 뭐가 있는지 보기(빼지는 않음)


## 흐름(읽는 순서)

1) 스택 개념 잡기  
2) 구현 1: 파이썬 리스트로 스택  
3) 구현 2: 배열 + 인덱스(top)로 스택  
4) 스택 응용: 괄호 검사  
5) 시스템 스택: 함수 호출/복귀  
6) 재귀: 왜 스택이랑 닮았나  
7) 피보나치: 중복 호출 → Memoization → DP  
8) 요약 + 미니 퀴즈


## 1. 핵심 개념

### 스택(Stack)
- **후입선출(LIFO, Last-In First-Out)**: 나중에 들어온 것이 먼저 나온다.
- 가능한 기본 연산
  - `push(x)`: x를 스택에 넣기
  - `pop()`: 가장 위(top)의 원소 꺼내기
  - `top/peek`: 가장 위 원소 확인(꺼내진 않음)

### 비유로 쉽게 이해하기
- **접시 더미**: 위에 올리고(push), 위에서 꺼낸다(pop).
- **브라우저 뒤로가기**: 방금 봤던 페이지부터 되돌아간다.
- **게임 되감기**: 마지막 행동부터 하나씩 되돌린다.

### 시간 복잡도 감각
- 보통 스택의 push/pop은 **O(1)** 로 처리된다.  
  (맨 끝/맨 위만 다루는 구조라서, 가운데를 건드릴 일이 없어요.)

![stack push/pop](<화면 캡처 2026-03-02 175028.jpg>)


## 2. 구현 1 — 파이썬 리스트로 스택

리스트의 맨 끝을 “스택의 top”으로 쓰면 구현이 단순해요.

- `append(x)`  → push
- `pop()`      → pop

아래 코드는 **원본 그대로**입니다.


In [1]:
s = []
def my_push(item):
    s.append(item)

In [2]:
# (추가) 나중에 같은 이름 함수가 다시 정의되기 때문에, 여기서 별칭을 만들어 둡니다.
my_push_list = my_push

### 간단 테스트(추가)

`my_push_list()`로 값을 넣고, 리스트 `s`에 잘 쌓였는지 확인해봅니다.


In [3]:
s.clear()
my_push_list(10)
my_push_list(20)
my_push_list(30)
print("s =", s)  # [10, 20, 30] (맨 오른쪽이 top)


s = [10, 20, 30]


### Pop(리스트 기반)

스택이 비어있는데 pop을 하면 안 되니까, 먼저 비었는지 확인합니다.  
아래 코드는 **원본 그대로**입니다.


In [4]:
def my_pop():
    if len(s) == 0:
        print('Undefflow!')
        return
    else:
        return s.pop()  # 리스트 s의 마지막 원소 삭제

In [5]:
# (추가) 리스트 기반 pop도 별칭으로 보관
my_pop_list = my_pop

#### 테스트(추가)

In [6]:
print("pop ->", my_pop_list())
print("pop ->", my_pop_list())
print("남은 s =", s)


pop -> 30
pop -> 20
남은 s = [10]


## 3. 구현 2 — 배열 + top 인덱스로 스택

리스트 대신 **크기가 정해진 배열**(리스트를 고정 크기처럼 사용)을 만들고,  
`top`이라는 숫자로 “현재 스택 맨 위 위치”를 직접 관리할 수도 있어요.

- `top = -1` 이면 **빈 스택**
- push할 때는 `top += 1` 하고 그 칸에 값을 넣는다
- pop할 때는 `stack[top]`을 꺼낸 뒤 `top -= 1` 한다

이 방식에서 자주 나오는 단어:
- **Overflow**: 더 넣으려고 했는데 공간이 없다(가득 찼다)
- **Underflow**: 꺼내려고 했는데 비어있다

아래 코드는 **원본 그대로**입니다.


In [1]:
def my_push(item, size):
    global top
    top += 1
    if top == size:
        print('Overflow!')
    else:
        stack[top] = item

In [2]:
# (추가) 배열 기반 push도 별칭으로 보관
my_push_arr = my_push

### Push 예시(원본 그대로)

In [3]:
size = 10
stack = [0] * size
top = -1

my_push(10, size)
top += 1
stack[top] = 20

### 같은 동작을 “함수 호출”로만 깔끔하게(추가)

원본 셀에서는 push를 한 번은 함수로, 한 번은 수동으로 했어요.  
헷갈리지 않게 보통은 **함수로만** 통일해서 씁니다.


In [10]:
# reset 후, 함수로만 push 해보기(추가)
size = 5
stack = [0] * size
top = -1

my_push_arr(10, size)
my_push_arr(20, size)
my_push_arr(30, size)
print("top =", top)
print("stack =", stack)


top = 2
stack = [10, 20, 30, 0, 0]


### Pop(배열 + top 기반)

아래 pop 함수는 **원본 그대로**입니다.


In [11]:
def my_pop():
    global top
    if top == -1:
        print('Underflow')
        return 0
    else:
        top -= 1
        return stack[top + 1]
    
print(my_pop())

30


In [12]:
# (추가) 배열 기반 pop도 별칭으로 보관
my_pop_arr = my_pop

#### 원본 예시 출력(원본 그대로)

In [13]:
if top > -1:
    top -= 1
    print(stack[top + 1])

20


#### 테스트(추가)

In [14]:
# reset 후 pop 흐름 확인(추가)
size = 5
stack = [0] * size
top = -1
my_push_arr(100, size)
my_push_arr(200, size)
my_push_arr(300, size)

print("pop ->", my_pop_arr())
print("pop ->", my_pop_arr())
print("pop ->", my_pop_arr())
print("pop(empty) ->", my_pop_arr())  # Underflow 기대


pop -> 300
pop -> 200
pop -> 100
Underflow
pop(empty) -> 0


## 4. 자주 하는 실수(실전에서 많이 나옴)

- **함수 이름이 덮어써짐**  
  노트북에서 `def my_push`를 다시 쓰면, 이전 `my_push`는 사라지고 **새 함수로 교체**됩니다.  
  그래서 위에서 `my_push_list`, `my_push_arr`처럼 별칭을 만들어 뒀어요.

- **Underflow / Overflow를 그냥 무시**  
  비어있는데 pop을 하거나, 가득 찼는데 push를 하면 값이 깨질 수 있어요.  
  최소한 “경고 출력”이라도 꼭 해두는 편이 안전합니다.

- **top 초기값(-1) 혼동**  
  `top=-1`은 “아직 아무 것도 없다”는 뜻.  
  첫 push가 되면 `top=0`이 됩니다.


## 5. 스택 응용 — 괄호 검사

괄호 문자열이 올바른지 확인할 때 스택이 자주 등장합니다.

### 아이디어(왜 스택?)
- 왼쪽 괄호를 만나면 “나중에 닫아야 할 숙제”로 생각하고 스택에 쌓아둔다.
- 오른쪽 괄호를 만나면 **가장 최근에 쌓인 왼쪽 괄호**와 짝이 맞는지 확인한다.
  - 최근 것이 먼저 처리되니까 스택(LIFO)이 딱 맞아요.

### 전략(단계)
1) 문자열을 왼쪽부터 한 글자씩 본다  
2) 여는 괄호면 push  
3) 닫는 괄호면 스택 top과 짝이 맞는지 확인 후 pop  
4) 끝까지 왔을 때 스택이 비어있으면 성공

### 의사코드
```text
for ch in s:
  if ch is opening:
    push(ch)
  else: # closing
    if stack empty: return False
    if top doesn't match: return False
    pop()
return stack empty
```

(아래 코드는 설명을 위한 **추가 코드**입니다.)


In [15]:
def is_valid_parentheses(s: str) -> bool:
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    opens = set(pairs.values())

    for ch in s:
        if ch in opens:
            stack.append(ch)
        else:
            if not stack:
                return False
            if stack[-1] != pairs.get(ch, None):
                return False
            stack.pop()
    return len(stack) == 0

tests = ["()[]{}", "([)]", "((()))", "(", "", "{[()()]}" ]
for t in tests:
    print(t, "->", is_valid_parentheses(t))


()[]{} -> True
([)] -> False
((())) -> True
( -> False
 -> True
{[()()]} -> True


## 6. 시스템 스택 — 함수 호출/복귀가 스택으로 관리되는 이유

프로그램이 함수 `A()` 안에서 `B()`를 부르고, `B()`가 끝나면 다시 `A()`로 돌아오죠.  
이때 “돌아갈 자리(복귀 주소)”와 “함수 실행에 필요한 정보(지역변수 등)”를 **스택에 차곡차곡** 저장합니다.

### 쉽게 말하면
- 함수 호출은 “잠깐 다른 일을 하러 가는 것”
- 돌아오려면 “어디까지 하다가 갔는지” 메모가 필요
- 그 메모를 스택에 올려두고, 끝나면 맨 위 메모부터 꺼내서(pop) 복귀한다

![함수 호출 흐름](<화면 캡처 2026-03-02 180756.jpg>)
![스택 프레임](<화면 캡처 2026-03-02 181046.jpg>)
![호출/복귀와 스택 변화](<화면 캡처 2026-03-02 181328.jpg>)


## 7. 재귀(Recursion) — “함수가 자기 자신을 부르는 구조”

재귀는 겉으로 보면 “함수가 자기 자신을 계속 호출”하는데,  
속을 보면 **함수 호출이 쌓이는 방식이 스택과 완전히 같습니다.**

### 재귀가 안전하게 동작하려면 2가지가 필요
1) **종료 조건**: 언제 멈출지(안 그러면 끝없이 호출)  
2) **문제를 줄이는 방식**: 호출할 때마다 입력이 작아져야 함

### 팩토리얼 그림으로 감 잡기
팩토리얼은 `n! = n × (n-1)!` 처럼 “작아지는 문제”로 바뀌어요.

![팩토리얼 재귀 흐름](<화면 캡처 2026-03-02 181642.jpg>)


In [16]:
def fact(n: int) -> int:
    if n <= 1:
        return 1
    return n * fact(n-1)

for i in range(1, 6):
    print(i, fact(i))


1 1
2 2
3 6
4 24
5 120


### 배열을 재귀로 훑기(그림 → 코드로 연결)

재귀 기본형을 이렇게 많이 씁니다:

- `f(i, N)` : 현재 위치 `i`에서 시작해서, 목표 `N`까지 처리

![재귀 기본형(접근 흐름)](<화면 캡처 2026-03-02 182031.jpg>)
![모든 원소 접근 예시](<화면 캡처 2026-03-02 182137.jpg>)

(아래 코드는 설명을 위한 **추가 코드**입니다.)


In [17]:
def print_all(arr, i=0):
    # 종료 조건: i가 끝(N)까지 가면 멈춤
    if i == len(arr):
        return
    print(arr[i])
    print_all(arr, i+1)

print_all([1, 2, 3])


1
2
3


### 배열에서 값 찾기(재귀)

- 끝까지 갔는데 못 찾으면 0
- 중간에 찾으면 1을 반환하고 더 내려가지 않음(바로 종료)

![못 찾는 경우 흐름](<화면 캡처 2026-03-02 182318.jpg>)
![찾는 경우 흐름](<화면 캡처 2026-03-02 182435.jpg>)

(아래 코드는 설명을 위한 **추가 코드**입니다.)


In [18]:
def find_value(arr, v, i=0):
    if i == len(arr):
        return 0
    if arr[i] == v:
        return 1
    return find_value(arr, v, i+1)

arr = [1, 2, 3, 4]
print("find 5 ->", find_value(arr, 5))
print("find 3 ->", find_value(arr, 3))


find 5 -> 0
find 3 -> 1


## 8. 피보나치 — 재귀의 “중복 호출” 문제를 눈으로 보기

피보나치 정의는 단순합니다:

- `F(0)=0`, `F(1)=1`
- `F(n)=F(n-1)+F(n-2)`

문제는 **같은 값을 여러 번 다시 계산**한다는 점이에요.

![중복 호출이 생기는 전개](<화면 캡처 2026-03-02 183340.jpg>)

아래 코드는 **원본 그대로**입니다.


In [19]:
def fibo(n):
    if n < 2:
        return n
    else:
        return fibo(n - 1) + fibo(n - 2)

### 호출이 얼마나 많은지 직접 세어보기(추가)

숫자가 조금만 커져도 호출 횟수가 폭발하는 걸 확인할 수 있어요.


In [20]:
def fibo_count(n: int):
    cnt = 0
    def f(x):
        nonlocal cnt
        cnt += 1
        if x < 2:
            return x
        return f(x-1) + f(x-2)
    return f(n), cnt

for n in [5, 10, 20]:
    val, calls = fibo_count(n)
    print(f"n={n:2d}  fib={val:5d}  calls={calls}")


n= 5  fib=    5  calls=15
n=10  fib=   55  calls=177
n=20  fib= 6765  calls=21891


## 9. 개선 1 — Memoization(메모이제이션)

이미 구한 `F(k)` 값을 배열에 저장해두고,  
다시 필요하면 계산하지 않고 바로 꺼내 쓰는 방법입니다.

- “같은 문제를 또 풀지 말고, 정답을 메모해두자”에 가깝습니다.

아래 코드는 **원본 그대로**인데, 실행을 위해 `n`을 미리 정해둘 필요가 있어요.  
그래서 `n`만 **추가 셀**에서 먼저 정의합니다.


In [21]:
# (추가) 원본 fibo1 셀이 동작하도록 n을 먼저 정의
n = 10

In [22]:
# memo를 위한 배열을 할당하고, 모두 0으로 초기화 한다.
# memo[0]을 0으로  memo[1]는 1로 초기화 한다.

def fibo1(n):
    if n >= 2 and memo[n] == 0:
        memo[n] = fibo1(n - 1) + fibo1(n - 2)
    return memo[n]

memo = [0] * (n + 1)
memo[0] = 0
memo[1] = 1

### 테스트(추가)

In [23]:
print("memo =", memo)
print("fibo1(10) =", fibo1(10))
print("memo(after) =", memo)


memo = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
fibo1(10) = 55
memo(after) = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


## 10. 개선 2 — DP(Dynamic Programming)

Memoization이 “필요할 때 저장”이라면,  
DP(반복형)는 “아예 작은 것부터 차례로 채우기”에 더 가깝습니다.

- 작은 값부터 `f[0], f[1], f[2] ...` 순서로 채우면  
  한 번 채운 값은 다시 계산할 일이 없어요.

아래 코드는 **원본 그대로**입니다.


In [24]:
def fibo2(n):
    f = [0] * (n + 1)
    f[0] = 0
    f[1] = 1
    for i in range(2, n + 1):
        f[i] = f[i - 1] + f[i - 2]
    
    return f[n]

### 테스트(추가)

In [25]:
for n in [0, 1, 2, 5, 10]:
    if n < 2:
        # fibo2 원본은 n>=1 가정으로 f[1]을 세팅하기 때문에,
        # 여기선 안전하게 분기해서 보여줍니다.
        print(n, "->", n)
    else:
        print(n, "->", fibo2(n))


0 -> 0
1 -> 1
2 -> 1
5 -> 5
10 -> 55


## 11. 복잡도 정리(한 번에 비교)

- 재귀 피보나치 `fibo(n)`  
  - 시간: 대략 **O(φ^n)** 수준(지수적으로 증가)  
  - 공간: 재귀 호출 스택 때문에 **O(n)**

- 메모이제이션 `fibo1(n)`  
  - 시간: **O(n)** (각 n을 한 번만 계산)  
  - 공간: **O(n)** (memo 테이블 + 호출 스택)

- DP 반복 `fibo2(n)`  
  - 시간: **O(n)**  
  - 공간: **O(n)** (테이블)

> “중복 계산을 없애면” 시간이 확 줄어든다 — 이게 오늘 핵심 포인트 중 하나입니다.


## 12. 요약(3줄)

- 스택은 **마지막에 넣은 것이 먼저 나오는(LIFO)** 구조이고, push/pop이 핵심이다.  
- 함수 호출/복귀, 재귀는 내부적으로 **호출 정보가 스택에 쌓였다가** 위에서부터 정리된다.  
- 피보나치 재귀는 중복 호출이 많아서 느리지만, **Memoization/DP로 O(n)**까지 개선된다.

### 미니 퀴즈(스스로 점검)
- [ ] 왜 괄호 검사는 “가장 최근에 열린 괄호”를 먼저 확인해야 할까?  
- [ ] `top=-1`이 의미하는 상태를 말로 설명할 수 있나?  
- [ ] `fibo(20)`이 느린 이유를 “중복 호출” 관점에서 설명할 수 있나?  
- [ ] `fibo1`과 `fibo2`의 차이를 한 문장으로 정리할 수 있나?
